# Phase 7: PatchCore Unsupervised Anomaly Detection & Ablation Study
### Towards Total Recall in Industrial Quality Inspection (Roth et al., CVPR 2022)

**Core Objectives:**
1. **Self-Contained Execution:** Automatically downloads the official MVTec AD Bottle dataset.
2. **Feature Extraction:** Frozen deep backbone (`wide_resnet50_2` / `resnet18`) multi-scale representation (Layers 2 & 3).
3. **Greedy k-Center Coreset Subsampling:** Compresses memory bank by 90% while guaranteeing minimax coverage.
4. **Official Benchmark Evaluation:** Image AUROC, Pixel AUROC, and Per-Region Overlap (AUPRO up to FPR <= 0.30).
5. **Industrial Ablation Study:**
   - ResNet-18 vs Wide-ResNet-50-2
   - 5% vs 10% vs 20% Coreset
   - Without vs With Gaussian Smoothing (sigma=0.0 vs sigma=4.0)
   - Max Patch Distance vs Top-5 Mean Patch Distance
6. **Artifact Export:** Checkpoints and plots automatically uploaded to your Hugging Face account.

## 1. Hardware Verification (Tesla T4 GPU)

In [ ]:
!nvidia-smi
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected! Go to Runtime -> Change runtime type -> Select T4 GPU.")

device = torch.device("cuda")
print(f"[Hardware] GPU Active   : {torch.cuda.get_device_name(0)}")
print(f"[Hardware] Total VRAM   : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Install Dependencies & Download MVTec Bottle Dataset

In [ ]:
!pip install -q torchvision scikit-learn scipy opencv-python matplotlib huggingface_hub

from pathlib import Path
from huggingface_hub import snapshot_download

dataset_dir = Path("datasets/raw/mvtec_ad/bottle")
if not (dataset_dir / "train" / "good").exists():
    print("[*] Downloading MVTec AD Bottle dataset from Hugging Face mirror...")
    snapshot_download(
        repo_id="foersben/mvtec-ad",
        repo_type="dataset",
        allow_patterns="bottle/*",
        local_dir="datasets/raw/mvtec_ad"
    )

train_good = list((dataset_dir / "train" / "good").glob("*.png"))
test_images = list((dataset_dir / "test").rglob("*.png"))
gt_masks = list((dataset_dir / "ground_truth").rglob("*.png"))

print(f"[OK] Dataset verified:")
print(f"     - Normal training images : {len(train_good)} (Expected: 209)")
print(f"     - Test benchmark images  : {len(test_images)} (Expected: 83)")
print(f"     - Ground truth masks     : {len(gt_masks)} (Expected: 63)")

## 3. Hugging Face Authentication

In [ ]:
import os
from huggingface_hub import login, HfApi

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

hf_username = None
if hf_token:
    login(token=hf_token, add_to_git_credential=True)
    api = HfApi()
    user_info = api.whoami()
    hf_username = user_info["name"]
    print(f"[Auth] Successfully logged into Hugging Face as: {hf_username}")
    
    REPO_NAME = "industrial-anomaly-patchcore"
    HF_REPO_ID = f"{hf_username}/{REPO_NAME}"
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True, private=True)
    print(f"[Hub] Target repository initialized: https://huggingface.co/{HF_REPO_ID}")
else:
    print("[Auth] Warning: HF_TOKEN not found in Colab Secrets. Models will save locally.")
    HF_REPO_ID = None

## 4. Define PatchCore Architecture, Coreset Sampler, & Metrics

In [ ]:
import math
import time
import json
from typing import Dict, List, Optional, Tuple, Union
import cv2
import numpy as np
import scipy.ndimage as ndimage
import matplotlib.pyplot as plt
from sklearn import metrics
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def load_image_tensor(path: Path, target_size=(224, 224)) -> torch.Tensor:
    bgr = cv2.imread(str(path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, target_size, interpolation=cv2.INTER_LINEAR)
    tensor = torch.from_numpy(resized).permute(2, 0, 1).float() / 255.0
    for c in range(3):
        tensor[c] = (tensor[c] - IMAGENET_MEAN[c]) / IMAGENET_STD[c]
    return tensor

class PatchCoreBackbone(nn.Module):
    def __init__(self, backbone_name="wide_resnet50_2", device="cuda"):
        super().__init__()
        self.device = torch.device(device)
        self.backbone_name = backbone_name.lower()
        
        if self.backbone_name == "resnet18":
            weights = models.ResNet18_Weights.IMAGENET1K_V1
            bb = models.resnet18(weights=weights)
            self.c2, self.c3 = 128, 256
        elif self.backbone_name == "wide_resnet50_2":
            weights = models.Wide_ResNet50_2_Weights.IMAGENET1K_V2
            bb = models.wide_resnet50_2(weights=weights)
            self.c2, self.c3 = 512, 1024
        else:
            raise ValueError(f"Unsupported: {backbone_name}")
            
        self.conv1, self.bn1, self.relu, self.maxpool = bb.conv1, bb.bn1, bb.relu, bb.maxpool
        self.layer1, self.layer2, self.layer3 = bb.layer1, bb.layer2, bb.layer3
        for p in self.parameters():
            p.requires_grad = False
        self.eval().to(self.device)
        
    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        feat2 = self.layer2(x)
        feat3 = self.layer3(feat2)
        return feat2, feat3

class GreedyCoresetSampler:
    def __init__(self, coreset_ratio=0.10, projection_dim=128, seed=42):
        self.coreset_ratio = coreset_ratio
        self.projection_dim = projection_dim
        self.seed = seed

    def sample(self, features: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        M, D = features.shape
        target_k = max(1, int(math.ceil(M * self.coreset_ratio)))
        if target_k >= M:
            return features, torch.arange(M, dtype=torch.long)

        torch.manual_seed(self.seed)
        if 0 < self.projection_dim < D:
            R = torch.randn(D, self.projection_dim, device=features.device, dtype=features.dtype)
            R /= math.sqrt(self.projection_dim)
            proj = torch.matmul(features, R)
        else:
            proj = features

        selected = [int(torch.randint(0, M, (1,)).item())]
        current_center = proj[selected[0]:selected[0] + 1]
        min_sq_dists = torch.sum((proj - current_center) ** 2, dim=1)

        for _ in range(1, target_k):
            next_idx = int(torch.argmax(min_sq_dists).item())
            selected.append(next_idx)
            new_center = proj[next_idx:next_idx + 1]
            new_sq = torch.sum((proj - new_center) ** 2, dim=1)
            min_sq_dists = torch.minimum(min_sq_dists, new_sq)

        sel_t = torch.tensor(selected, dtype=torch.long, device=features.device)
        return features[sel_t], sel_t

class PatchCore:
    def __init__(self, backbone_name="wide_resnet50_2", coreset_ratio=0.10, device="cuda"):
        self.device = torch.device(device)
        self.backbone_name = backbone_name
        self.coreset_ratio = coreset_ratio
        self.backbone = PatchCoreBackbone(backbone_name, device=self.device)
        self.sampler = GreedyCoresetSampler(coreset_ratio=coreset_ratio)
        self.memory_bank: Optional[torch.Tensor] = None

    def extract_patch_features(self, images: torch.Tensor):
        with torch.no_grad():
            f2, f3 = self.backbone(images)
            H2, W2 = f2.shape[2], f2.shape[3]
            f3_up = F.interpolate(f3, size=(H2, W2), mode="bilinear", align_corners=False)
            concat = torch.cat([f2, f3_up], dim=1)
            pooled = F.avg_pool2d(concat, kernel_size=3, stride=1, padding=1)
            B, D, _, _ = pooled.shape
            patches = pooled.permute(0, 2, 3, 1).reshape(B, H2 * W2, D)
            return patches, (H2, W2)

    def fit(self, dataloader: DataLoader):
        all_feats = []
        self.backbone.eval()
        with torch.no_grad():
            for batch in dataloader:
                batch = batch.to(self.device)
                patches, _ = self.extract_patch_features(batch)
                all_feats.append(patches.reshape(-1, patches.shape[-1]).cpu())
        full = torch.cat(all_feats, dim=0)
        coreset, _ = self.sampler.sample(full)
        self.memory_bank = coreset.to(self.device)
        return self.memory_bank.shape[0]

    def predict_patches(self, images: torch.Tensor, score_method="max", top_k=5, chunk_size=256):
        patches, (H2, W2) = self.extract_patch_features(images)
        B, P, D = patches.shape
        flat = patches.reshape(B * P, D)
        N = flat.shape[0]
        K = self.memory_bank.shape[0]

        min_dists = torch.empty(N, device=self.device, dtype=torch.float32)
        m_norm_sq = torch.sum(self.memory_bank ** 2, dim=1, keepdim=True).t()

        for start in range(0, N, chunk_size):
            end = min(start + chunk_size, N)
            q = flat[start:end]
            q_norm_sq = torch.sum(q ** 2, dim=1, keepdim=True)
            dot = torch.matmul(q, self.memory_bank.t())
            sq = torch.clamp(q_norm_sq + m_norm_sq - 2.0 * dot, min=0.0)
            min_sq, _ = torch.min(sq, dim=1)
            min_dists[start:end] = torch.sqrt(min_sq)

        patch_dists = min_dists.reshape(B, P)
        if score_method.lower() == "top_k":
            k_val = min(max(1, top_k), P)
            topk_vals, _ = torch.topk(patch_dists, k=k_val, dim=1, largest=True)
            image_scores = torch.mean(topk_vals, dim=1)
        else:
            image_scores, _ = torch.max(patch_dists, dim=1)
        return patch_dists, image_scores, (H2, W2)

    def compute_anomaly_maps(self, patch_dists, spatial_hw, output_hw, sigma=4.0):
        B, P = patch_dists.shape
        H2, W2 = spatial_hw
        grid = patch_dists.reshape(B, 1, H2, W2)
        upsampled = F.interpolate(grid, size=output_hw, mode="bilinear", align_corners=False).squeeze(1).cpu().numpy()
        if sigma <= 0.0:
            return upsampled
        smoothed = np.empty_like(upsampled)
        for i in range(B):
            smoothed[i] = ndimage.gaussian_filter(upsampled[i], sigma=sigma)
        return smoothed

def compute_metrics(y_true, y_scores, gt_masks, a_maps, max_fpr=0.30):
    img_auroc = float(metrics.roc_auc_score(y_true, y_scores))
    
    y_true_flat = np.concatenate([m.ravel() > 0 for m in gt_masks]).astype(np.int32)
    y_score_flat = np.concatenate([m.ravel() for m in a_maps]).astype(np.float32)
    step = max(1, len(y_true_flat) // 2_000_000)
    pix_auroc = float(metrics.roc_auc_score(y_true_flat[::step], y_score_flat[::step]))
    
    # AUPRO
    defect_regions = []
    total_normal_px = 0
    for idx, mask in enumerate(gt_masks):
        bin_m = (mask > 0).astype(np.uint8)
        total_normal_px += int(np.sum(bin_m == 0))
        num_labels, labels = cv2.connectedComponents(bin_m, connectivity=8)
        for l in range(1, num_labels):
            comp = (labels == l)
            defect_regions.append((idx, comp, int(np.sum(comp))))

    all_s = np.concatenate([m.ravel() for m in a_maps])
    thresholds = np.linspace(float(np.max(all_s)), float(np.min(all_s)), 150)
    fpr_list, pro_list = [0.0], [0.0]
    
    for th in thresholds:
        preds = [m >= th for m in a_maps]
        fp = sum(int(np.sum(p & ~(gt > 0))) for p, gt in zip(preds, gt_masks))
        fpr = fp / max(1, total_normal_px)
        pro_vals = [int(np.sum(preds[i] & comp)) / max(1, area) for i, comp, area in defect_regions]
        pro = float(np.mean(pro_vals)) if pro_vals else 0.0
        fpr_list.append(fpr)
        pro_list.append(pro)
        if fpr >= max_fpr:
            break

    fpr_arr, pro_arr = np.array(fpr_list), np.array(pro_list)
    valid = fpr_arr <= max_fpr
    fpr_v, pro_v = fpr_arr[valid], pro_arr[valid]
    if len(fpr_v) > 0 and fpr_v[-1] < max_fpr:
        fpr_v = np.append(fpr_v, max_fpr)
        pro_v = np.append(pro_v, pro_v[-1])
    aupro = float(np.trapezoid(pro_v, fpr_v) / max_fpr) if len(fpr_v) >= 2 else 0.0
    
    # F1 Optimal
    best_f1, best_th = 0.0, float(np.median(y_scores))
    for th in np.percentile(y_scores, np.linspace(10, 90, 80)):
        p_bin = (y_scores >= th).astype(int)
        f1 = metrics.f1_score(y_true, p_bin, zero_division=0)
        if f1 > best_f1:
            best_f1, best_th = f1, float(th)
            
    return img_auroc, pix_auroc, aupro, best_th, best_f1

## 5. Run Baseline PatchCore Training on Tesla T4

In [ ]:
print("=" * 70)
print("PHASE 7: RUNNING PATCHCORE BASELINE TRAINING ON COLAB T4 GPU")
print("=" * 70)

class BottleDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return load_image_tensor(self.paths[i])

train_paths = sorted(list((dataset_dir / "train" / "good").glob("*.png")))
train_loader = DataLoader(BottleDataset(train_paths), batch_size=16, shuffle=False)

baseline_model = PatchCore(backbone_name="wide_resnet50_2", coreset_ratio=0.10, device="cuda")
t0 = time.time()
n_protos = baseline_model.fit(train_loader)
fit_sec = time.time() - t0
print(f"[Fit] Memory bank fitted in {fit_sec:.2f}s with {n_protos} prototypes.")

# Test evaluation
test_samples = []
test_dir = dataset_dir / "test"
gt_dir = dataset_dir / "ground_truth"
for folder in sorted(test_dir.iterdir()):
    if not folder.is_dir(): continue
    is_defect = 0 if folder.name == "good" else 1
    for img_p in sorted(folder.glob("*.png")):
        mask_p = gt_dir / folder.name / f"{img_p.stem}_mask.png" if is_defect else None
        test_samples.append((img_p, mask_p, is_defect, folder.name))

y_true, y_scores, gt_masks, a_maps = [], [], [], []
t_inf_start = time.time()
for img_p, mask_p, is_defect, dtype in test_samples:
    img_t = load_image_tensor(img_p).unsqueeze(0).to("cuda")
    dists, scores, shp = baseline_model.predict_patches(img_t, score_method="max")
    amap = baseline_model.compute_anomaly_maps(dists, shp, output_hw=(224, 224), sigma=4.0)[0]
    
    if mask_p and mask_p.exists():
        m_gray = cv2.imread(str(mask_p), cv2.IMREAD_GRAYSCALE)
        m_bin = (cv2.resize(m_gray, (224, 224), interpolation=cv2.INTER_NEAREST) > 0).astype(np.uint8)
    else:
        m_bin = np.zeros((224, 224), dtype=np.uint8)
        
    y_true.append(is_defect)
    y_scores.append(float(scores[0].item()))
    gt_masks.append(m_bin)
    a_maps.append(amap)

mean_lat = ((time.time() - t_inf_start) / len(test_samples)) * 1000.0
img_auc, pix_auc, aupro, opt_th, f1 = compute_metrics(y_true, y_scores, gt_masks, a_maps)

print("\n" + "*" * 55)
print("             OFFICIAL BENCHMARK RESULTS            ")
print("*" * 55)
print(f"Backbone Architecture : Wide-ResNet-50-2")
print(f"Image-level AUROC     : {img_auc:.4f} (Target >= 0.980)")
print(f"Pixel-level AUROC     : {pix_auc:.4f} (Target >= 0.960)")
print(f"AUPRO (FPR <= 0.30)   : {aupro:.4f} (Target >= 0.920)")
print(f"Optimal Threshold     : {opt_th:.4f}")
print(f"F1-Score at Threshold : {f1:.4f}")
print(f"Mean Latency / Image  : {mean_lat:.2f} ms")
print("*" * 55)

## 6. Run the 4-Way Industrial Ablation Study
Evaluates:
1. **Backbone:** ResNet-18 vs Wide-ResNet-50-2
2. **Coreset Ratio:** 5% vs 10% vs 20%
3. **Spatial Filtering:** Without vs With Gaussian Smoothing (sigma=0.0 vs sigma=4.0)
4. **Scoring Function:** Max Patch Distance vs Top-5 Mean Patch Distance

In [ ]:
ablation_configs = [
    {"name": "Baseline (Wide-ResNet-50-2, 10% Coreset, Smoothed, Max)", "backbone": "wide_resnet50_2", "coreset": 0.10, "sigma": 4.0, "score": "max"},
    {"name": "Backbone: ResNet-18 (Lightweight Edge Model)", "backbone": "resnet18", "coreset": 0.10, "sigma": 4.0, "score": "max"},
    {"name": "Coreset: 5% Subsampling (~8k prototypes)", "backbone": "resnet18", "coreset": 0.05, "sigma": 4.0, "score": "max"},
    {"name": "Coreset: 20% Subsampling (~32k prototypes)", "backbone": "resnet18", "coreset": 0.20, "sigma": 4.0, "score": "max"},
    {"name": "Filtering: Raw Bilinear (No Gaussian, sigma=0.0)", "backbone": "resnet18", "coreset": 0.10, "sigma": 0.0, "score": "max"},
    {"name": "Scoring: Top-5 Mean Patch Distance (Speckle Robust)", "backbone": "resnet18", "coreset": 0.10, "sigma": 4.0, "score": "top_k"},
]

ablation_results = []
print("=" * 70)
print("STARTING ABLATION SUITE ON TESLA T4")
print("=" * 70)

for cfg in ablation_configs:
    print(f"\n--> Evaluating: {cfg['name']}...")
    m = PatchCore(backbone_name=cfg["backbone"], coreset_ratio=cfg["coreset"], device="cuda")
    t_fit_start = time.time()
    protos = m.fit(train_loader)
    fit_d = time.time() - t_fit_start
    
    y_t, y_s, gts, amaps = [], [], [], []
    t_inf = time.time()
    for img_p, mask_p, is_def, _ in test_samples:
        img_t = load_image_tensor(img_p).unsqueeze(0).to("cuda")
        dists, scores, shp = m.predict_patches(img_t, score_method=cfg["score"], top_k=5)
        amap = m.compute_anomaly_maps(dists, shp, output_hw=(224, 224), sigma=cfg["sigma"])[0]
        
        if mask_p and mask_p.exists():
            mg = cv2.imread(str(mask_p), cv2.IMREAD_GRAYSCALE)
            mb = (cv2.resize(mg, (224, 224), interpolation=cv2.INTER_NEAREST) > 0).astype(np.uint8)
        else:
            mb = np.zeros((224, 224), dtype=np.uint8)
        y_t.append(is_def)
        y_s.append(float(scores[0].item()))
        gts.append(mb)
        amaps.append(amap)
        
    lat = ((time.time() - t_inf) / len(test_samples)) * 1000.0
    im_auc, px_auc, aup, _, _ = compute_metrics(y_t, y_s, gts, amaps)
    mem_mb = (m.memory_bank.element_size() * m.memory_bank.nelement()) / (1024 ** 2)
    
    res = {
        "name": cfg["name"],
        "backbone": cfg["backbone"],
        "coreset": f"{cfg['coreset']*100:.0f}%",
        "sigma": cfg["sigma"],
        "score": cfg["score"],
        "img_auroc": im_auc,
        "pix_auroc": px_auc,
        "aupro": aup,
        "mem_mb": mem_mb,
        "latency_ms": lat,
        "fit_sec": fit_d
    }
    ablation_results.append(res)
    print(f"    Image AUROC: {im_auc:.4f} | Pixel AUROC: {px_auc:.4f} | AUPRO: {aup:.4f} | Latency: {lat:.1f}ms")

print("\n" + "=" * 70)
print("ABLATION STUDY SUMMARY TABLE")
print("=" * 70)
print(f"{'Experiment':<45} | {'Img AUROC':<10} | {'Pix AUROC':<10} | {'AUPRO':<8} | {'Latency':<8}")
print("-" * 90)
for r in ablation_results:
    print(f"{r['name'][:45]:<45} | {r['img_auroc']:<10.4f} | {r['pix_auroc']:<10.4f} | {r['aupro']:<8.4f} | {r['latency_ms']:<7.1f}ms")

## 7. Generate Comparative Plots & Inspection Heatmaps

In [ ]:
# 1. Plot Ablation Comparison Bar Charts
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
names_short = [r["name"].split("(")[0].strip() for r in ablation_results]
x = np.arange(len(names_short))
width = 0.25

axes[0].bar(x - width, [r["img_auroc"] for r in ablation_results], width, label="Image AUROC", color="#1f77b4")
axes[0].bar(x, [r["pix_auroc"] for r in ablation_results], width, label="Pixel AUROC", color="#2ca02c")
axes[0].bar(x + width, [r["aupro"] for r in ablation_results], width, label="AUPRO", color="#ff7f0e")
axes[0].set_ylabel("Metric Score")
axes[0].set_title("Ablation Metrics Comparison (Image AUROC, Pixel AUROC, AUPRO)", fontweight="bold")
axes[0].set_xticks(x)
axes[0].set_xticklabels(names_short, rotation=35, ha="right", fontsize=9)
axes[0].set_ylim(0.85, 1.01)
axes[0].legend()
axes[0].grid(axis="y", linestyle="--", alpha=0.7)

axes[1].bar(names_short, [r["latency_ms"] for r in ablation_results], color="#9467bd", width=0.5)
axes[1].set_ylabel("Latency (ms / image)")
axes[1].set_title("Inference Latency Benchmark", fontweight="bold")
axes[1].set_xticklabels(names_short, rotation=35, ha="right", fontsize=9)
axes[1].grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plot_comp_path = "patchcore_ablation_comparison.png"
plt.savefig(plot_comp_path, dpi=200, bbox_inches="tight")
plt.show()

# 2. Plot Inspection Heatmaps for 4 Test Classes
rep_indices = [0] # good
for idx, (_, _, is_def, dtype) in enumerate(test_samples):
    if is_def and dtype not in [test_samples[i][3] for i in rep_indices]:
        rep_indices.append(idx)
    if len(rep_indices) >= 4:
        break

fig, ax = plt.subplots(len(rep_indices), 3, figsize=(12, 3.5 * len(rep_indices)))
for row, idx in enumerate(rep_indices):
    img_p, _, _, dtype = test_samples[idx]
    raw = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
    raw_res = cv2.resize(raw, (224, 224))
    gt_m = gt_masks[idx]
    a_m = a_maps[idx]
    
    ax[row, 0].imshow(raw_res)
    ax[row, 0].set_title(f"Test Sample: {dtype}", fontweight="bold")
    ax[row, 0].axis("off")
    
    ax[row, 1].imshow(gt_m, cmap="gray")
    ax[row, 1].set_title("Ground Truth Mask")
    ax[row, 1].axis("off")
    
    im = ax[row, 2].imshow(a_m, cmap="jet")
    ax[row, 2].set_title(f"PatchCore Anomaly Heatmap (Score: {y_scores[idx]:.3f})")
    ax[row, 2].axis("off")
    plt.colorbar(im, ax=ax[row, 2], fraction=0.046, pad=0.04)

plt.tight_layout()
plot_heat_path = "patchcore_mvtec_bottle_evaluation.png"
plt.savefig(plot_heat_path, dpi=200, bbox_inches="tight")
plt.show()

## 8. Export Model Checkpoint & Upload to Hugging Face Hub

In [ ]:
os.makedirs("models/checkpoints", exist_ok=True)
ckpt_file = "models/checkpoints/patchcore_bottle_wide_resnet50_2.pt"
torch.save({
    "backbone_name": "wide_resnet50_2",
    "coreset_ratio": 0.10,
    "memory_bank": baseline_model.memory_bank.cpu(),
    "threshold": opt_th
}, ckpt_file)
print(f"[Save] Checkpoint saved locally to: {ckpt_file}")

metrics_file = "models/checkpoints/patchcore_benchmark_metrics.json"
with open(metrics_file, "w") as f:
    json.dump({
        "image_auroc": round(img_auc, 4),
        "pixel_auroc": round(pix_auc, 4),
        "aupro": round(aupro, 4),
        "optimal_threshold": round(opt_th, 4),
        "f1_score": round(f1, 4),
        "mean_latency_ms": round(mean_lat, 2),
        "ablations": ablation_results
    }, f, indent=2)
print(f"[Save] Benchmark metrics saved locally to: {metrics_file}")

if HF_REPO_ID and hf_token:
    print(f"\n[Hub] Uploading artifacts to Hugging Face Hub: {HF_REPO_ID}...")
    try:
        api.upload_file(path_or_fileobj=ckpt_file, path_in_repo="checkpoints/patchcore_bottle_wide_resnet50_2.pt", repo_id=HF_REPO_ID, repo_type="model")
        api.upload_file(path_or_fileobj=metrics_file, path_in_repo="metrics/patchcore_benchmark_metrics.json", repo_id=HF_REPO_ID, repo_type="model")
        api.upload_file(path_or_fileobj=plot_heat_path, path_in_repo="visuals/patchcore_mvtec_bottle_evaluation.png", repo_id=HF_REPO_ID, repo_type="model")
        api.upload_file(path_or_fileobj=plot_comp_path, path_in_repo="visuals/patchcore_ablation_comparison.png", repo_id=HF_REPO_ID, repo_type="model")
        print(f"[Hub] All artifacts uploaded successfully to: https://huggingface.co/{HF_REPO_ID}")
    except Exception as e:
        print(f"[Hub] Upload note: {e}")
else:
    print("[Hub] Skipping upload (HF_TOKEN not provided).")